# 🫁 Pneumonia Detection from Chest X-Rays
**Custom CNN vs. Transfer Learning (DenseNet121)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kart747/pneumonia-detection/blob/main/pneumonia_detection.ipynb)

Binary classification of chest X-rays into **NORMAL** vs **PNEUMONIA** using the
[Kaggle Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) dataset (5,863 images).

**Pipeline**
1. Download dataset from Kaggle (API key / kagglehub / manual)
2. EDA – class balance & sample images
3. Re-split train/val (the official val set has only 16 images)
4. Model 1 – **Custom CNN** trained from scratch
5. Model 2 – **DenseNet121** (ImageNet weights, the CheXNet backbone): frozen-base training → fine-tuning
6. Evaluation on the held-out test set: accuracy, precision, recall, F1, ROC-AUC, confusion matrix
7. **Grad-CAM** to see where the model looks
8. Save models + results for GitHub

> ⚙️ **Runtime → Change runtime type → T4 GPU** before running.

## 1. Setup

In [ ]:
import os, random, glob, json, shutil, subprocess, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, accuracy_score, precision_score, recall_score, f1_score)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "None (training will be slow!)")

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ---- config ----
IMG_SIZE      = 224
BATCH_SIZE    = 32
VAL_FRACTION  = 0.15
CNN_EPOCHS    = 25     # early stopping usually ends earlier
TL_EPOCHS     = 10     # phase 1: frozen DenseNet121
FT_EPOCHS     = 10     # phase 2: fine-tuning
FINE_TUNE_LAST_N = 60  # number of DenseNet layers to unfreeze
CLASS_NAMES   = ["NORMAL", "PNEUMONIA"]   # label 0, label 1
PRETRAINED_WEIGHTS = "imagenet"

WORK_DIR    = Path("/content") if Path("/content").exists() else Path.cwd()
DATA_DIR    = WORK_DIR / "data"
MODELS_DIR  = WORK_DIR / "models"
RESULTS_DIR = WORK_DIR / "results"
for d in (DATA_DIR, MODELS_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 2. Download the dataset from Kaggle
Pick **one** method in `DOWNLOAD_METHOD`:

| Method | What you need |
|---|---|
| `"api"` *(default)* | Your `kaggle.json` – Kaggle → **Settings → API → Create New Token**. You'll be asked to upload it. |
| `"kagglehub"` | Nothing (public dataset) – uses the `kagglehub` library. |
| `"manual"` | Download the zip from Kaggle yourself and upload it to `/content/data/` (or Google Drive) and set `MANUAL_ZIP`. |

In [ ]:
DOWNLOAD_METHOD = "api"          # "api" | "kagglehub" | "manual"
KAGGLE_DATASET  = "paultimothymooney/chest-xray-pneumonia"
MANUAL_ZIP      = DATA_DIR / "archive.zip"   # used only for "manual"

def download_with_api():
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    token = kaggle_dir / "kaggle.json"
    if not token.exists():
        from google.colab import files
        print("Upload your kaggle.json (Kaggle → Settings → API → Create New Token)")
        uploaded = files.upload()
        name = next(iter(uploaded))
        token.write_bytes(uploaded[name])
    os.chmod(token, 0o600)
    subprocess.run(["pip", "install", "-q", "kaggle"], check=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", KAGGLE_DATASET,
                    "-p", str(DATA_DIR), "--unzip", "-q"], check=True)
    return DATA_DIR

def download_with_kagglehub():
    subprocess.run(["pip", "install", "-q", "kagglehub"], check=True)
    import kagglehub
    return Path(kagglehub.dataset_download(KAGGLE_DATASET))

def download_manual():
    with zipfile.ZipFile(MANUAL_ZIP) as z:
        z.extractall(DATA_DIR)
    return DATA_DIR

def find_data_root(base):
    """The zip nests folders (chest_xray/chest_xray, __MACOSX...). Find the one with train/ and test/."""
    for p in sorted(Path(base).rglob("train")):
        if "__MACOSX" in p.parts:
            continue
        root = p.parent
        if (root / "test" / "NORMAL").is_dir() and (p / "PNEUMONIA").is_dir():
            return root
    raise FileNotFoundError(f"Could not find train/test folders under {base}")

downloaders = {"api": download_with_api, "kagglehub": download_with_kagglehub, "manual": download_manual}
try:
    DATA_ROOT = find_data_root(DATA_DIR)          # already downloaded? skip
    print("Dataset already present.")
except FileNotFoundError:
    DATA_ROOT = find_data_root(downloaders[DOWNLOAD_METHOD]())
print("DATA_ROOT =", DATA_ROOT)

## 3. Explore the data
The official split is `train / val / test`, but **val has only 16 images**, which makes
validation metrics useless. We merge train+val and carve out a stratified 15 % validation split.
The test set stays untouched for the final evaluation.

In [ ]:
IMG_EXTS = {".jpeg", ".jpg", ".png"}

def list_split(split):
    paths, labels = [], []
    for label, cls in enumerate(CLASS_NAMES):
        for p in sorted((DATA_ROOT / split / cls).glob("*")):
            if p.suffix.lower() in IMG_EXTS:
                paths.append(str(p)); labels.append(label)
    return paths, labels

orig = {s: list_split(s) for s in ["train", "val", "test"]}
counts = pd.DataFrame({s: pd.Series(l).value_counts().reindex([0, 1], fill_value=0).values
                       for s, (_, l) in orig.items()}, index=CLASS_NAMES)
print("Original split:"); display(counts)

# merge train + val, then stratified re-split
all_paths  = orig["train"][0] + orig["val"][0]
all_labels = orig["train"][1] + orig["val"][1]
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels, test_size=VAL_FRACTION, stratify=all_labels, random_state=SEED)
test_paths, test_labels = orig["test"]

split_counts = pd.DataFrame({
    "train": np.bincount(train_labels, minlength=2),
    "val":   np.bincount(val_labels,   minlength=2),
    "test":  np.bincount(test_labels,  minlength=2)}, index=CLASS_NAMES)
print("Split used for training:"); display(split_counts)

ax = split_counts.T.plot(kind="bar", figsize=(7, 4), color=["#4C9BE8", "#E8744C"], rot=0)
ax.set_title("Images per class"); ax.set_ylabel("count")
for c in ax.containers: ax.bar_label(c)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "class_distribution.png", dpi=120); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for row, label in enumerate([0, 1]):
    picks = random.sample([p for p, l in zip(train_paths, train_labels) if l == label], 5)
    for ax, p in zip(axes[row], picks):
        ax.imshow(plt.imread(p), cmap="gray"); ax.set_title(CLASS_NAMES[label]); ax.axis("off")
plt.suptitle("Sample training images"); plt.tight_layout()
plt.savefig(RESULTS_DIR / "samples.png", dpi=120); plt.show()

## 4. Input pipeline
* `tf.data` reads JPEGs → 3 channels → resize to 224×224 (pixel range 0–255; each model normalises internally).
* **Class imbalance** (~3:1 pneumonia:normal) is handled with **class weights**.
* **Augmentation** lives inside the models (small rotation/zoom/shift/contrast, active only during training).
  No horizontal flips – the heart is on the left, flipping makes anatomically wrong X-rays.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img, tf.cast(label, tf.float32)

def make_ds(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds(train_paths, train_labels, shuffle=True)
val_ds   = make_ds(val_paths, val_labels)
test_ds  = make_ds(test_paths, test_labels)

n0, n1 = np.bincount(train_labels)
class_weight = {0: (n0 + n1) / (2 * n0), 1: (n0 + n1) / (2 * n1)}
print("class_weight:", {CLASS_NAMES[k]: round(v, 3) for k, v in class_weight.items()})

augment = keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.1),
], name="augment")

METRICS = [keras.metrics.BinaryAccuracy(name="accuracy"), keras.metrics.AUC(name="auc"),
           keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")]

def callbacks(name):
    return [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7, verbose=1),
        keras.callbacks.ModelCheckpoint(str(MODELS_DIR / f"{name}.keras"), monitor="val_loss", save_best_only=True),
    ]

## 5. Model 1 – Custom CNN (from scratch)
4 conv blocks (Conv-BN-ReLU ×2 → MaxPool → Dropout), filters 32→64→128→256, then
Global Average Pooling → Dense(128) → sigmoid.

In [ ]:
def conv_block(x, filters, drop):
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
    x = layers.MaxPooling2D()(x)
    return layers.Dropout(drop)(x)

def build_custom_cnn():
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = augment(inputs)
    x = layers.Rescaling(1. / 255)(x)
    for f, d in [(32, 0.1), (64, 0.15), (128, 0.2), (256, 0.25)]:
        x = conv_block(x, f, d)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name="custom_cnn")

cnn = build_custom_cnn()
cnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=METRICS)
cnn.summary()

In [ ]:
cnn_hist = cnn.fit(train_ds, validation_data=val_ds, epochs=CNN_EPOCHS,
                   class_weight=class_weight, callbacks=callbacks("custom_cnn"))
cnn_history = pd.DataFrame(cnn_hist.history)

## 6. Model 2 – Transfer learning with DenseNet121
DenseNet121 is the backbone of Stanford's **CheXNet**. Two phases:
1. **Feature extraction** – freeze the ImageNet-pretrained base, train only the new head (lr = 1e-3).
2. **Fine-tuning** – unfreeze the last `FINE_TUNE_LAST_N` layers and train with a tiny lr (1e-5)
   so the pretrained features adapt to X-rays without being destroyed.
BatchNorm layers stay in inference mode (`training=False`) – standard practice when fine-tuning.

In [ ]:
def build_densenet():
    base = keras.applications.DenseNet121(include_top=False, weights=PRETRAINED_WEIGHTS,
                                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = augment(inputs)
    # DenseNet's ImageNet preprocessing: scale to [0,1] then normalise with ImageNet mean/std
    x = layers.Rescaling(1. / 255, name="rescale")(x)
    x = layers.Normalization(mean=[0.485, 0.456, 0.406],
                             variance=[0.229 ** 2, 0.224 ** 2, 0.225 ** 2], name="imagenet_norm")(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="head_dropout")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="pred")(x)
    return keras.Model(inputs, outputs, name="densenet121_tl"), base

tl_model, tl_base = build_densenet()
tl_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=METRICS)
print(f"Trainable params (phase 1): {sum(np.prod(w.shape) for w in tl_model.trainable_weights):,}")

tl_hist1 = tl_model.fit(train_ds, validation_data=val_ds, epochs=TL_EPOCHS,
                        class_weight=class_weight, callbacks=callbacks("densenet121_tl"))

In [ ]:
# ---- Phase 2: fine-tune the top of DenseNet121 ----
tl_base.trainable = True
for layer in tl_base.layers[:-FINE_TUNE_LAST_N]:
    layer.trainable = False
for layer in tl_base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

tl_model.compile(optimizer=keras.optimizers.Adam(1e-5), loss="binary_crossentropy", metrics=METRICS)
print(f"Trainable params (phase 2): {sum(np.prod(w.shape) for w in tl_model.trainable_weights):,}")

start = len(tl_hist1.history["loss"])
tl_hist2 = tl_model.fit(train_ds, validation_data=val_ds, epochs=start + FT_EPOCHS, initial_epoch=start,
                        class_weight=class_weight, callbacks=callbacks("densenet121_tl"))
tl_history = pd.concat([pd.DataFrame(tl_hist1.history), pd.DataFrame(tl_hist2.history)], ignore_index=True)
FT_START = start

## 7. Training curves

In [ ]:
def plot_history(h, title, fname, ft_start=None):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    for ax, m in zip(axes, ["loss", "accuracy", "auc"]):
        ax.plot(h[m], label="train"); ax.plot(h[f"val_{m}"], label="val")
        if ft_start is not None:
            ax.axvline(ft_start - 0.5, ls="--", c="gray", label="fine-tune start")
        ax.set_title(m); ax.set_xlabel("epoch"); ax.legend()
    fig.suptitle(title); plt.tight_layout()
    plt.savefig(RESULTS_DIR / fname, dpi=120); plt.show()

plot_history(cnn_history, "Custom CNN", "history_custom_cnn.png")
plot_history(tl_history, "DenseNet121 (transfer learning)", "history_densenet121.png", ft_start=FT_START)

## 8. Evaluation on the test set (624 unseen images)
In medical screening **recall (sensitivity) for PNEUMONIA** matters most – a missed pneumonia
is worse than a false alarm – so look at recall and the confusion matrix, not just accuracy.

In [ ]:
y_true = np.array(test_labels)

def evaluate(model, name, threshold=0.5):
    probs = model.predict(test_ds, verbose=0).ravel()
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    print(f"\n===== {name} =====")
    print(classification_report(y_true, preds, labels=[0, 1], target_names=CLASS_NAMES, digits=4, zero_division=0))
    return probs, {
        "model": name,
        "accuracy":  accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall (sensitivity)": recall_score(y_true, preds, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "f1": f1_score(y_true, preds, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probs) if len(set(y_true)) > 1 else float("nan"),
    }

cnn_probs, cnn_metrics = evaluate(cnn, "Custom CNN")
tl_probs,  tl_metrics  = evaluate(tl_model, "DenseNet121 (TL)")

results = pd.DataFrame([cnn_metrics, tl_metrics]).set_index("model").round(4)
results.to_csv(RESULTS_DIR / "metrics.csv")
display(results)
print(results.to_markdown())   # paste this into the README

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, probs) in zip(axes[:2], [("Custom CNN", cnn_probs), ("DenseNet121 (TL)", tl_probs)]):
    cm = confusion_matrix(y_true, (probs >= 0.5).astype(int), labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(f"{name} – confusion matrix"); ax.set_xlabel("predicted"); ax.set_ylabel("actual")

for name, probs in [("Custom CNN", cnn_probs), ("DenseNet121 (TL)", tl_probs)]:
    fpr, tpr, _ = roc_curve(y_true, probs)
    axes[2].plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_true, probs):.3f})")
axes[2].plot([0, 1], [0, 1], "k--", lw=1)
axes[2].set_title("ROC curve"); axes[2].set_xlabel("False positive rate"); axes[2].set_ylabel("True positive rate")
axes[2].legend()
plt.tight_layout(); plt.savefig(RESULTS_DIR / "confusion_roc.png", dpi=120); plt.show()

ax = results[["accuracy", "precision", "recall (sensitivity)", "specificity", "f1", "roc_auc"]].T.plot(
    kind="bar", figsize=(10, 4), rot=0, ylim=(0, 1.05))
ax.set_title("Custom CNN vs DenseNet121 – test metrics")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "model_comparison.png", dpi=120); plt.show()

## 9. Grad-CAM – where is the model looking?
Grad-CAM weights DenseNet's last 7×7 feature map by the gradient of the prediction,
giving a heat-map of the regions that drove the decision. Good models should focus on the lungs.

In [ ]:
def gradcam(img):
    """img: (H, W, 3) float 0-255. Returns (heatmap HxW in [0,1], prob)."""
    x = tf.convert_to_tensor(img[None, ...], tf.float32)
    x = tl_model.get_layer("imagenet_norm")(tl_model.get_layer("rescale")(x))
    with tf.GradientTape() as tape:
        fmap = tl_base(x, training=False)              # (1, 7, 7, 1024)
        tape.watch(fmap)
        prob = tl_model.get_layer("pred")(tl_model.get_layer("gap")(fmap))
        # explain the predicted class: P(pneumonia) if predicted pneumonia, else P(normal)
        target = tf.where(prob >= 0.5, prob, 1.0 - prob)
    grads = tape.gradient(target, fmap)
    weights = tf.reduce_mean(grads, axis=(1, 2))           # (1, 1024)
    cam = tf.nn.relu(tf.reduce_sum(fmap * weights[:, None, None, :], axis=-1))[0]
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE))[..., 0]
    return cam.numpy(), float(prob[0, 0])

idx = [i for i in range(len(test_paths)) if test_labels[i] == 0][:3] + \
      [i for i in range(len(test_paths)) if test_labels[i] == 1][:3]
fig, axes = plt.subplots(2, len(idx), figsize=(4 * len(idx), 8))
for col, i in enumerate(idx):
    img, _ = load_image(test_paths[i], test_labels[i])
    img = img.numpy()
    cam, prob = gradcam(img)
    pred = CLASS_NAMES[int(prob >= 0.5)]
    axes[0, col].imshow(img.astype("uint8")); axes[0, col].axis("off")
    axes[0, col].set_title(f"true: {CLASS_NAMES[test_labels[i]]}")
    axes[1, col].imshow(img.astype("uint8")); axes[1, col].imshow(cam, cmap="jet", alpha=0.4)
    axes[1, col].axis("off"); axes[1, col].set_title(f"pred: {pred} ({prob:.2f})")
plt.suptitle("Grad-CAM (DenseNet121)"); plt.tight_layout()
plt.savefig(RESULTS_DIR / "gradcam.png", dpi=120); plt.show()

## 10. Predict on any X-ray

In [ ]:
def predict_xray(path, model=None):
    model = model or tl_model
    img, _ = load_image(path, 0)
    prob = float(model.predict(img[None, ...], verbose=0)[0, 0])
    label = CLASS_NAMES[int(prob >= 0.5)]
    conf = prob if prob >= 0.5 else 1 - prob
    return label, conf

for p in [test_paths[0], test_paths[-1]]:
    label, conf = predict_xray(p)
    print(f"{Path(p).name:40s} → {label} ({conf:.1%})")

In [ ]:
cnn.save(MODELS_DIR / "custom_cnn.keras")
tl_model.save(MODELS_DIR / "densenet121_tl.keras")
for f in sorted(MODELS_DIR.glob("*.keras")):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:6.1f} MB")

shutil.make_archive(str(WORK_DIR / "results"), "zip", RESULTS_DIR)
print("Saved results/ (plots + metrics.csv) and results.zip")

## 11. (Optional) Download results & push to GitHub
* **Notebook:** *File → Save a copy in GitHub* → pick your repo. The saved notebook keeps all outputs (plots, metrics) = your proof.
* **Plots/metrics:** run the cell below to download `results.zip`, unzip it into the repo's `results/` folder and commit.
* Model files are large – keep them out of git (see `.gitignore`) or attach them to a GitHub Release.

In [ ]:
try:
    from google.colab import files
    files.download(str(WORK_DIR / "results.zip"))
except ImportError:
    print("Not running in Colab – results are in", RESULTS_DIR)

## (Optional) Live demo with Gradio
Creates a temporary public link where you can upload an X-ray and get a prediction + Grad-CAM.

In [ ]:
RUN_DEMO = False   # set True to launch
if RUN_DEMO:
    subprocess.run(["pip", "install", "-q", "gradio"], check=True)
    import gradio as gr

    def demo_fn(image):
        img = tf.image.resize(tf.convert_to_tensor(image, tf.float32), (IMG_SIZE, IMG_SIZE)).numpy()
        if img.shape[-1] == 1: img = np.repeat(img, 3, -1)
        cam, prob = gradcam(img[..., :3])
        heat = (plt.cm.jet(cam)[..., :3] * 255 * 0.4 + img[..., :3] * 0.6).astype("uint8")
        return {"PNEUMONIA": prob, "NORMAL": 1 - prob}, heat

    gr.Interface(fn=demo_fn, inputs=gr.Image(type="numpy", label="Chest X-ray"),
                 outputs=[gr.Label(label="Prediction"), gr.Image(label="Grad-CAM")],
                 title="Pneumonia Detection (DenseNet121)").launch(share=True)

---
⚠️ *Educational project – not a medical device. Predictions must not be used for diagnosis.*